In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.stats import spearmanr, pearsonr
import pandas as pd
import scipy.io
import pickle
import os
import warnings
from scipy.stats import pearsonr, zscore, spearmanr
import seaborn as sns
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, surface
from nilearn.plotting import plot_surf_stat_map
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")

os.chdir('/gpfs/milgram/project/chun/jk2992/socialaha/') # change to your folder path

nroi_cor, nroi_sub = 100, 16
nroi = nroi_cor + nroi_sub

''' setting '''
flist = {}
flist[1] = ['sub-1001', 'sub-1005', 'sub-1008', 'sub-1011', 'sub-1014', 'sub-1017', 'sub-1020', 'sub-1023', 'sub-1026', 'sub-1029', 'sub-1033', 'sub-1039']
flist[2] = ['sub-2006', 'sub-2009', 'sub-2012', 'sub-2015', 'sub-2018', 'sub-2021', 'sub-2024', 'sub-2027', 'sub-2034', 'sub-2038', 'sub-2040'] # 'sub-2030'
flist[3] = ['sub-3004', 'sub-3007', 'sub-3013', 'sub-3016', 'sub-3019', 'sub-3022', 'sub-3025', 'sub-3031', 'sub-3037', 'sub-3041'] # 'sub-3010', 'sub-3028'
tasklist = ['01','02','03','04','05','06','07','08','09','10']
# sub-2030, sub-3010, sub-3028: large head motion participants
# sub-1023 task-03: only movie watching portion was recorded
nsubj = len(flist[1])+len(flist[2])+len(flist[3])

In [2]:
hrf = 3
window = 1
dissimilarity_allgroups_all_tp = {}
for groupid in range(1,3+1):
    file_name = "./data/brain/loaded_BOLD/ROIsum_combined_mask_g"+str(groupid)+".pkl"
    with open(file_name, "rb") as file:
        loaded_data = pickle.load(file)
    print('running group',groupid)
    this_group = []
    for roi in range(1,nroi+1):
        print('running ROI',roi)
        for sub in range(len(flist[groupid])):
            dissim_bysub = []
            for run in range(1,10): # skipping one because there is no prior thoughts
                if run == 6: # skipping run 7
                    pass
                else:
                    tst = pd.read_csv('./data/brain/events/'+flist[groupid][sub]+'_task-'+tasklist[run]+'_events.tsv', sep='\t')
                    TRs = int(tst['onset'][4]+tst['duration'][4]) #get the movie duration
                    this_run = loaded_data[sub][run][roi][:,hrf:TRs+hrf]
                    dissimilarity = []
                    for tr in range(window,this_run.shape[1]-window):
                        dissim = 1 - pearsonr(this_run[:,tr-window],this_run[:,tr])[0]
                        dissimilarity.append(float(dissim))
                    dissim_bysub.append(np.array(dissimilarity))
            dissimilarity_allgroups_all_tp[groupid,roi,sub] = dissim_bysub

running group 1
running ROI 1
running ROI 2
running ROI 3
running ROI 4
running ROI 5
running ROI 6
running ROI 7
running ROI 8
running ROI 9
running ROI 10
running ROI 11
running ROI 12
running ROI 13
running ROI 14
running ROI 15
running ROI 16
running ROI 17
running ROI 18
running ROI 19
running ROI 20
running ROI 21
running ROI 22
running ROI 23
running ROI 24
running ROI 25
running ROI 26
running ROI 27
running ROI 28
running ROI 29
running ROI 30
running ROI 31
running ROI 32
running ROI 33
running ROI 34
running ROI 35
running ROI 36
running ROI 37
running ROI 38
running ROI 39
running ROI 40
running ROI 41
running ROI 42
running ROI 43
running ROI 44
running ROI 45
running ROI 46
running ROI 47
running ROI 48
running ROI 49
running ROI 50
running ROI 51
running ROI 52
running ROI 53
running ROI 54
running ROI 55
running ROI 56
running ROI 57
running ROI 58
running ROI 59
running ROI 60
running ROI 61
running ROI 62
running ROI 63
running ROI 64
running ROI 65
running ROI 66
run

In [3]:
# Save the similarity matrix
np.save('./data/brain/pattern_shift/'+str(window)+'TR_nearbytp.npy', dissimilarity_allgroups_all_tp, allow_pickle=True)